# LISS-IV Cloud Removal — Collaborative Training with DagsHub & MLflow

This notebook allows team members to train **U-Net** and **Pix2Pix Conditional GAN (cGAN)** models on free Google Colab GPUs (T4), streaming all parameters, losses, metrics, and checkpoints to a **centralized, shared DagsHub MLflow dashboard** (`https://dagshub.com/Mnthn2326/liss4-cloud-removal`).

> **⚠️ IMPORTANT TEAM RULE:** Do NOT run the entire notebook from top to bottom!
> - **Everyone runs:** Steps 1, 2, 3, and 4.
> - **In Step 5:** ONLY run your assigned Track (A, B, C, or D). Skip the others!
> - **In Step 6 & 7:** Run inference on your generated checkpoint and download it.

## Step 1: GPU Hardware Check (All Teammates)

In [ ]:
import torch
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"[OK] GPU Detected: {torch.cuda.get_device_name(0)}")
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
else:
    print("[WARNING] GPU not detected. Go to Runtime -> Change runtime type -> Select T4 GPU.")

## Step 2: Clone Repository & Install Dependencies (All Teammates)

In [ ]:
import os

if not os.path.exists("liss4-cloud-removal"):
    !git clone https://github.com/Mnthn2326/liss4-cloud-removal.git

%cd liss4-cloud-removal

!pip install -q -e .
!pip install -q dagshub mlflow
print("[OK] Environment setup complete.")

## Step 3: Connect to Shared DagsHub MLflow (All Teammates)
> **🚨 CHANGE `TEAMMATE_NAME` BELOW BEFORE RUNNING!** Do not leave it as default.

In [ ]:
import os
import dagshub

# >>> CHANGE THIS TO YOUR ACTUAL NAME <<<
TEAMMATE_NAME = "YOUR_NAME_HERE"  # e.g., "Manthan", "Alice", "Bob", etc.
os.environ["MLFLOW_USER"] = TEAMMATE_NAME

# Initialize DagsHub MLflow tracking (Prompts for authorization token on first run)
dagshub.init(repo_owner="Mnthn2326", repo_name="liss4-cloud-removal", mlflow=True)

print(f"[OK] Connected to DagsHub MLflow! Logging runs under user: {TEAMMATE_NAME}")
print(f"Tracking URI: {os.environ.get('MLFLOW_TRACKING_URI')}")

## Step 4: Acquire Dataset (All Teammates)
Verifies and downloads the SEN12MS-CR paired optical training subset if not present (~1 minute).

In [ ]:
from pathlib import Path

manifest_path = Path("data/raw/sen12mscr/manifest.json")
if not manifest_path.is_file():
    print("Downloading SEN12MS-CR paired optical dataset...")
    !python -m cloud_removal.acquire_sen12mscr
else:
    print("[OK] Dataset manifest already verified on disk.")

---
# Step 5: Execute Assigned Training Track

⚠️ **RUN ONLY THE SINGLE CELL ASSIGNED TO YOU! DO NOT RUN ALL TRACKS!**

### 👤 Track A — Teammate 1: Standard Full U-Net (7.76M Parameters)
Trains the full-capacity U-Net baseline on RGB using Compound L1 + SSIM loss.

In [ ]:
# Teammate 1 ONLY runs this cell (Cell 11):
!python -m cloud_removal.train --config configs/train_unet.yaml

### 👤 Track B — Teammate 2: Compact Mobile U-Net (1.94M Parameters)
Trains the lightweight, high-throughput U-Net designed for edge/API serving.

In [ ]:
# Teammate 2 ONLY runs this cell (Cell 13):
!python -m cloud_removal.train --config configs/train_compact_unet.yaml

### 👤 Track C — Teammate 3: Pix2Pix Conditional GAN (cGAN)
Trains U-Net Generator + PatchGAN Discriminator with Adversarial + L1 Content Loss for edge sharpness.

In [ ]:
# Teammate 3 ONLY runs this cell (Cell 15):
!python -m cloud_removal.train_cgan --config configs/train_cgan.yaml

### 👤 Track D — Teammate 4: ISRO LISS-IV False-Color Composite (FCC)
Trains cross-sensor aligned model on Sentinel-2 NIR (B08), Red (B04), and Green (B03) matching LISS-IV.

In [ ]:
# Teammate 4 ONLY runs this cell (Cell 17):
!python -m cloud_removal.train_cgan --config configs/train_fcc_cgan.yaml

---
## Step 6: Visual Inference & Comparison (All Teammates)
Generates side-by-side comparison figure (`inference_result.png`) for validation sample `0`.

In [ ]:
from IPython.display import Image, display
from pathlib import Path

# Auto-detect the best checkpoint saved by your run
candidate_checkpoints = [
    "checkpoints/best_unet.pt",
    "checkpoints/compact/best_compact_unet.pt",
    "checkpoints/cgan/best_cgan_generator.pt",
    "checkpoints/fcc_cgan/best_fcc_cgan_generator.pt",
]
active_checkpoint = next((c for c in candidate_checkpoints if Path(c).is_file()), None)

if active_checkpoint:
    print(f"Evaluating checkpoint: {active_checkpoint}")
    output_fig = "reports/figures/colab_inference_result.png"
    !python -m cloud_removal.infer --checkpoint $active_checkpoint --split val --index 0 --output $output_fig
    display(Image(output_fig))
else:
    print("No checkpoint found. Did your training run finish?")

## Step 7: Download Checkpoint to Local Machine (All Teammates)
Downloads your trained `.pt` weights file directly to your local computer.

In [ ]:
from google.colab import files

if active_checkpoint and Path(active_checkpoint).is_file():
    print(f"Downloading {active_checkpoint}...")
    files.download(active_checkpoint)
else:
    print("No checkpoint available for download.")